# Week 64, Day 5: The import system

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w064d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sys, types

config = types.ModuleType("config")
exec("DEBUG = True\nLEVEL = 3\n", config.__dict__)
print(config.DEBUG, config.LEVEL, type(config).__name__)

sys.modules["config"] = config
import config as again
print(again is config)
del sys.modules["config"]

import json
print(json.__spec__.name, "json" in sys.modules)

In [ ]:
import importlib, sys

mod = importlib.import_module("os.path")
top = __import__("os.path")
print(mod.__name__ in ("posixpath", "ntpath"), top.__name__)

import importlib.util
spec = importlib.util.find_spec("json")
print(spec.name, spec.submodule_search_locations is not None, type(spec.loader).__name__)
print([type(f).__name__ for f in sys.meta_path][:3])

In [ ]:
import importlib, sys, types

source = "VERSION = 1\ndef hello():\n    return 'hi v1'\n"
mod = types.ModuleType("greetings")
mod.__spec__ = None
exec(source, mod.__dict__)
sys.modules["greetings"] = mod

from greetings import hello                 # a reference to the v1 function
exec(source.replace("1", "2"), mod.__dict__) # what reload does: re-run the code in the same module
import greetings
print(greetings.VERSION, greetings.hello(), hello())   # the old reference keeps the old function
del sys.modules["greetings"]

In [ ]:
import importlib, importlib.abc, importlib.util, sys

class MemoryImporter(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def __init__(self, sources):
        self.sources = sources
    def find_spec(self, fullname, path=None, target=None):
        if fullname not in self.sources:
            return None                                       # not ours: let others try
        return importlib.util.spec_from_loader(fullname, self, origin="memory")
    def create_module(self, spec):
        return None                                           # the default module object
    def exec_module(self, module):
        exec(compile(self.sources[module.__name__], "<memory:" + module.__name__ + ">", "exec"), module.__dict__)

importer = MemoryImporter({"tax": "RATE = 0.2\ndef add_tax(p):\n    return round(p * (1 + RATE), 2)\n"})
sys.meta_path.insert(0, importer)
try:
    import tax
    print(tax.add_tax(10), tax.__spec__.origin, tax.__file__ if hasattr(tax, "__file__") else "no file")
finally:
    sys.meta_path.remove(importer)
    sys.modules.pop("tax", None)

In [ ]:
import importlib, sys

print(importlib.util.resolve_name(".prices", "shop"), importlib.util.resolve_name("..utils", "shop.cart"))
import json, json.decoder
print(json.__path__ is not None, json.decoder.__package__, hasattr(json.decoder, "__path__"))
import importlib.util as iu
print(iu.find_spec("json").submodule_search_locations is not None, iu.find_spec("json.decoder").submodule_search_locations)

In [ ]:
import importlib, importlib.abc, importlib.util, sys

sources = {
    "orders": "import customers\nDISCOUNT = 0.1\ndef price(p):\n    return p * (1 - DISCOUNT)\n",
    "customers": "from orders import DISCOUNT\n",
}

class Mem(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, name, path=None, target=None):
        return importlib.util.spec_from_loader(name, self) if name in sources else None
    def create_module(self, spec):
        return None
    def exec_module(self, module):
        exec(sources[module.__name__], module.__dict__)

finder = Mem()
sys.meta_path.insert(0, finder)
try:
    import orders
except ImportError as e:
    print("ImportError:", str(e).split(" (")[0])
finally:
    sys.meta_path.remove(finder)
    for name in sources:
        sys.modules.pop(name, None)

> **Key idea.** import checks sys.modules first; otherwise it asks each finder on sys.meta_path for a module spec (name, loader, origin, submodule_search_locations), creates the module, puts it in sys.modules before running it, runs it with loader.exec_module, and binds names. importlib.import_module imports by string and returns the module itself. sys.modules is the cache: modules run once, reload re-runs code in the same object, and deleting an entry forces a fresh module. Extend imports with a class having find_spec (returning None for names it doesn't handle) and exec_module, inserted at the front of sys.meta_path and removed afterwards. Packages have __path__; relative imports resolve against __package__. Circular imports see partly initialised modules: restructure, import inside functions, or use import a and a.name.

### Exercise: Modules from source

Write `module_from_source(name, source)`: a new module object (with `types.ModuleType`) named `name`, with `__file__` set to `"<memory>"`, whose code has been executed in its namespace; it must **not** be added to `sys.modules`. Then `public_names(module)`: the sorted list of names in the module's namespace that don't start with `_`, or, if the module defines `__all__`, exactly the names in `__all__` (sorted).

In [ ]:
import sys
import types

def module_from_source(name, source):
    ___

def public_names(module):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fromsource():\n    \"module objects\"\n    import sys\n    m = module_from_source(\"shapes\", \"import math\\nPI2 = math.pi * 2\\n_hidden = 1\\ndef area(r):\\n    return math.pi * r * r\\n\")\n    assert m.__name__ == \"shapes\" and m.__file__ == \"<memory>\" and round(m.area(1), 2) == 3.14\n    assert \"shapes\" not in sys.modules and public_names(m) == [\"PI2\", \"area\", \"math\"]\n    n = module_from_source(\"api\", \"__all__ = ['b', 'a']\\na = b = c = 1\\n\")\n    assert public_names(n) == [\"a\", \"b\"]\n    try:\n        module_from_source(\"bad\", \"x = 1 / 0\")\n        assert False\n    except ZeroDivisionError:\n        pass")

### Exercise: Working with sys.modules

Write `loaded(prefix)`: the sorted names in `sys.modules` that are `prefix` itself or start with `prefix + "."`. Then `forget(prefix)`: remove all those entries and return how many were removed, and `fresh_import(name)`: forget the module and its submodules, import it again with `importlib.import_module`, and return the **new** module object.

In [ ]:
import importlib
import sys

def loaded(prefix):
    ___

def forget(prefix):
    ___

def fresh_import(name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"the module cache\"\n    import sys, types, colorsys\n    old = colorsys\n    new = fresh_import(\"colorsys\")\n    assert new is not old and sys.modules[\"colorsys\"] is new and new.rgb_to_hsv(1, 0, 0)[0] == 0\n    for name in (\"fakepkg\", \"fakepkg.a\", \"fakepkg.b\", \"fakepkgextra\"):\n        sys.modules[name] = types.ModuleType(name)\n    try:\n        assert loaded(\"fakepkg\") == [\"fakepkg\", \"fakepkg.a\", \"fakepkg.b\"]\n        assert forget(\"fakepkg\") == 3 and loaded(\"fakepkg\") == [] and \"fakepkgextra\" in sys.modules\n    finally:\n        for name in (\"fakepkg\", \"fakepkg.a\", \"fakepkg.b\", \"fakepkgextra\"):\n            sys.modules.pop(name, None)")

### Exercise: An importer from memory

Write `MemoryImporter(sources)`, a meta path finder and loader for plain modules (no packages yet) whose source code is in the dict `sources` (module name to source). It records each module it **executes** in its list `executed`. Then write a context manager `using(importer)` that puts the importer at the **front** of `sys.meta_path` for the duration of the block and afterwards removes it and every module that was imported during the block from `sys.modules` (even after an error). Names the importer doesn't know must still import normally.

In [ ]:
import importlib.abc
import importlib.util
import sys
from contextlib import contextmanager

class MemoryImporter(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def __init__(self, sources):
        ___

@contextmanager
def using(importer):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memory():\n    \"a custom importer\"\n    import importlib, sys\n    sources = {\"greet\": \"def hello(name):\\n    return 'hello ' + name\\n\", \"maths2\": \"import greet\\nTWO = 2\\n\"}\n    imp = MemoryImporter(sources)\n    with using(imp) as same:\n        assert same is imp\n        m = importlib.import_module(\"maths2\")\n        assert m.TWO == 2 and m.greet.hello(\"ada\") == \"hello ada\"\n        assert importlib.import_module(\"greet\") is m.greet and imp.executed == [\"maths2\", \"greet\"]\n        assert m.__spec__.loader is imp and sys.meta_path[0] is imp\n        import string\n        assert string.ascii_lowercase[0] == \"a\"\n        try:\n            importlib.import_module(\"nowhere_at_all\")\n            assert False\n        except ModuleNotFoundError:\n            pass\n    assert imp not in sys.meta_path and \"maths2\" not in sys.modules and \"greet\" not in sys.modules\n    try:\n        with using(imp):\n            importlib.import_module(\"greet\")\n            raise ValueError\n    except ValueError:\n        pass\n    assert \"greet\" not in sys.modules and imp.executed == [\"maths2\", \"greet\", \"greet\"]")

### Exercise: Watching imports

Write `ImportWatcher`, a meta path finder that never provides modules (its `find_spec` always returns `None`) but records each name it's asked about in `requested`, in order, without duplicates. Then `imports_triggered(func)`: run `func()` with a fresh watcher at the front of `sys.meta_path` and return its `requested` list (always removing the watcher). Names already in `sys.modules` never reach finders, so they won't appear.

In [ ]:
import sys

class ImportWatcher:
    def __init__(self):
        ___

def imports_triggered(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_watch():\n    \"finders are asked in order\"\n    import sys, importlib\n    def load():\n        sys.modules.pop(\"colorsys\", None)\n        import colorsys\n        import sys as again\n        return colorsys\n    names = imports_triggered(load)\n    assert names == [\"colorsys\"]\n    assert not any(isinstance(f, ImportWatcher) for f in sys.meta_path)\n    def nothing():\n        import sys\n    assert imports_triggered(nothing) == []\n    def fails():\n        importlib.import_module(\"not_a_real_module_xyz\")\n    try:\n        imports_triggered(fails)\n        assert False\n    except ModuleNotFoundError:\n        pass\n    assert not any(isinstance(f, ImportWatcher) for f in sys.meta_path)")

### Exercise: Simulating circular imports

Model a program's imports as a dict from module name to the list of `(kind, target, name)` steps its top level runs, in order: `("import", "b", None)` for `import b`, `("from", "b", "x")` for `from b import x`, and `("define", None, "x")` for defining `x`. Write `simulate_import(modules, start)` following Python's rules: a module enters the cache (with no names yet) **before** its steps run; importing a cached module does nothing more; `from b import x` imports `b` and then fails if `x` isn't defined in `b` **yet**. Return the list of modules in the order they **finished** running, or raise `ImportError(f"cannot import name {x!r} from partially initialized module {b!r}")`.

In [ ]:
def simulate_import(modules, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cycles():\n    \"partially initialised modules\"\n    ok = {\"app\": [(\"import\", \"models\", None), (\"define\", None, \"main\")],\n          \"models\": [(\"import\", \"app\", None), (\"define\", None, \"User\")]}\n    assert simulate_import(ok, \"app\") == [\"models\", \"app\"]\n    bad = {\"orders\": [(\"import\", \"customers\", None), (\"define\", None, \"DISCOUNT\")],\n           \"customers\": [(\"from\", \"orders\", \"DISCOUNT\")]}\n    try:\n        simulate_import(bad, \"orders\")\n        assert False\n    except ImportError as e:\n        assert str(e) == \"cannot import name 'DISCOUNT' from partially initialized module 'orders'\"\n    fixed = {\"orders\": [(\"define\", None, \"DISCOUNT\"), (\"import\", \"customers\", None)],\n             \"customers\": [(\"from\", \"orders\", \"DISCOUNT\")]}\n    assert simulate_import(fixed, \"orders\") == [\"customers\", \"orders\"]\n    diamond = {\"a\": [(\"import\", \"b\", None), (\"import\", \"c\", None)], \"b\": [(\"import\", \"d\", None)],\n               \"c\": [(\"from\", \"d\", \"x\")], \"d\": [(\"define\", None, \"x\")]}\n    assert simulate_import(diamond, \"a\") == [\"d\", \"b\", \"c\", \"a\"]")

### Bug hunt: __import__ returns the top package

`__import__("os.path")` imports `os.path` but returns the **top-level** package `os`, which has no `join` of its own, so `mod.join` raises `AttributeError`. Use `importlib.import_module`, which returns the submodule itself, so this prints `a/b`.

In [ ]:
import importlib

mod = __import__("os.path")
print(mod.join("a", "b").replace("\\", "/"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"import_module\"\n    expect_output(\"a/b\")")

### Bug hunt: A loader that ran the code somewhere else

`exec_module` runs the source in a new, empty dict instead of the module's own namespace, so the module comes out empty and `units.METRE` raises `AttributeError`. Run the code in `module.__dict__` so this prints `1.0`.

In [ ]:
import importlib.abc, importlib.util, sys

class Loader(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, name, path=None, target=None):
        return importlib.util.spec_from_loader(name, self) if name == "units" else None
    def create_module(self, spec):
        return None
    def exec_module(self, module):
        exec("METRE = 1.0", {})

finder = Loader()
sys.meta_path.insert(0, finder)
try:
    import units
    print(units.METRE)
finally:
    sys.meta_path.remove(finder)
    sys.modules.pop("units", None)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the module's namespace\"\n    expect_output(\"1.0\")")

### Bug hunt: A finder that claims everything

This finder returns a spec for **every** name it's asked about, so even `import string` comes from it, as an empty module, and `string.digits` raises `AttributeError`. Return `None` for names the finder doesn't provide, so the normal finders handle them and this prints `0123456789`.

In [ ]:
import importlib.abc, importlib.util, sys

class Greedy(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    provides = {"extras": "VALUE = 1"}
    def find_spec(self, name, path=None, target=None):
        return importlib.util.spec_from_loader(name, self)
    def create_module(self, spec):
        return None
    def exec_module(self, module):
        exec(self.provides.get(module.__name__, ""), module.__dict__)

finder = Greedy()
sys.modules.pop("string", None)
sys.meta_path.insert(0, finder)
try:
    import string
    print(string.digits)
finally:
    sys.meta_path.remove(finder)
    sys.modules.pop("string", None)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return None\"\n    expect_output(\"0123456789\")")

### Puzzle: A minimal finder

Arrange the lines of a finder method that only provides modules it knows about. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the first place import looks: "sys.path", "sys.meta_path" or "sys.modules"
q = ___   # True or False: a module is added to sys.modules before its code runs
r = ___   # what __import__("a.b") returns: "a" or "a.b"
s = ___   # the attribute that makes a module a package
t = ___   # importlib.util.resolve_name(".models", "shop")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"sys.modules\" and q is True and r == \"a\", \"p, q and r\"\n    assert s == \"__path__\" and t == \"shop.models\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: packages from memory

Extend the memory importer to support **packages**. Sources are keyed by path: `"shop/__init__"` is the package `shop`, `"shop/cart"` is the module `shop.cart`, `"util"` is the top-level module `util`. Write `PackageImporter(sources)` so that:

- `find_spec` finds both packages (making a spec with `is_package=True`, so the module gets a `__path__`) and modules, and returns `None` for anything else;
- relative imports inside packages work (`from . import cart`, `from .prices import price`), since the import system resolves them from `__package__`;
- each module's `__file__` is `"memory:" + key` (set it in `exec_module`);
- `executed` records each module's full name as it starts executing;
- `tree()` returns a dict from each package name to the sorted list of its direct submodule names (full names), including packages with no submodules, for everything in `sources`.

Also write `using(importer)` as in the earlier exercise.

In [ ]:
import importlib.abc
import importlib.util
import sys
from contextlib import contextmanager

class PackageImporter(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def __init__(self, sources):
        ___

@contextmanager
def using(importer):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_packages():\n    \"packages and relative imports\"\n    import importlib, sys\n    src = {\n        \"shop/__init__\": \"from . import cart\\nNAME = 'shop'\\n\",\n        \"shop/cart\": \"from .prices import price\\ndef total(items):\\n    return sum(price(i) for i in items)\\n\",\n        \"shop/prices\": \"TABLE = {'tea': 3, 'jam': 4}\\ndef price(item):\\n    return TABLE[item]\\n\",\n        \"shop/admin/__init__\": \"\",\n        \"shop/admin/reports\": \"from ..prices import TABLE\\nCOUNT = len(TABLE)\\n\",\n        \"util\": \"X = 1\\n\",\n    }\n    imp = PackageImporter(src)\n    with using(imp):\n        shop = importlib.import_module(\"shop\")\n        assert shop.cart.total([\"tea\", \"jam\"]) == 7 and shop.NAME == \"shop\"\n        assert hasattr(shop, \"__path__\") and not hasattr(shop.cart, \"__path__\")\n        assert imp.executed == [\"shop\", \"shop.cart\", \"shop.prices\"]\n        reports = importlib.import_module(\"shop.admin.reports\")\n        assert reports.COUNT == 2 and reports.__package__ == \"shop.admin\"\n        assert shop.__file__ == \"memory:shop/__init__\" and reports.__file__ == \"memory:shop/admin/reports\"\n        import util\n        assert util.X == 1 and util.__file__ == \"memory:util\"\n        try:\n            importlib.import_module(\"shop.missing\")\n            assert False\n        except ModuleNotFoundError:\n            pass\n    assert \"shop\" not in sys.modules and \"shop.cart\" not in sys.modules\n    assert imp.tree() == {\"shop\": [\"shop.admin\", \"shop.cart\", \"shop.prices\"], \"shop.admin\": [\"shop.admin.reports\"]}")

## Warm-up

A module is just an object with a namespace dict. Putting one in `sys.modules` was enough for `import config` to find it, without any file: the very first thing `import` does is look in that dict.

## What import really does

`import shop.cart` (and every other form of import) calls `importlib.__import__`, which, for each part of the dotted name in turn (`shop`, then `shop.cart`):

1. **Checks `sys.modules`.** If the name is there, that module is used, and nothing else happens. That's why a module's code runs only once per program.
2. **Finds a spec.** It asks each **finder** in `sys.meta_path`, in order, `find_spec(name, path)`. The first to return a **module spec** wins; if all return `None`, the result is `ModuleNotFoundError`.
3. **Creates the module** with the spec's **loader** (`loader.create_module(spec)`, usually returning `None` to get a plain module object), sets its attributes (`__name__`, `__spec__`, `__file__`, `__package__`, `__path__` for packages, `__loader__`) and **puts it in `sys.modules` before running it**.
4. **Executes the module**: `loader.exec_module(module)` runs the module's code with the module's `__dict__` as its globals.
5. **Binds names**: `import shop.cart` binds `shop` in your namespace; `from shop import cart` binds `cart`; and the submodule is also set as an attribute of its parent (`shop.cart`).

If executing the module raises, the module is removed from `sys.modules` again, so a later import retries.

`importlib.import_module("shop.cart")` does steps 1 to 4 for a name held in a string and returns the module itself (whereas `__import__("shop.cart")` returns the top package, `shop`).

## sys.modules: the cache

`sys.modules` is a plain dict from module name to module object, and the import system trusts it completely:

- Importing an already-imported module is a dict lookup: cheap, and the module's code doesn't run again.
- **Changing a module's source file doesn't affect a running program**, because the old module object stays in `sys.modules`. `importlib.reload(module)` re-runs the module's code **in the same module object** (so existing references see the new definitions), but objects created from the old code (instances of old classes, functions imported with `from ... import`) keep the old versions.
- **Removing** an entry (`del sys.modules[name]`) makes the next import load the module afresh, as a **new** object; anything still holding the old module keeps it. Tests sometimes do this to get a fresh module.
- You can **insert** your own objects, as the warm-up did: test frameworks use this to replace a module with a fake.

## Finders, loaders and specs

The machinery is split into small, replaceable parts:

- A **finder** answers "can you provide this module?". Finders on **`sys.meta_path`** are asked first, for every import. CPython's defaults are, in order: `BuiltinImporter` (modules compiled into the interpreter, such as `sys`), `FrozenImporter` (modules frozen into the executable) and `PathFinder`, which searches the folders in **`sys.path`** (and, for submodules, the package's `__path__`) using **path entry finders** for ordinary files, `.pyc` files, C extensions and zip files.
- A **module spec** (`importlib.machinery.ModuleSpec`) is what a finder returns: the module's `name`, its `loader`, its `origin` (often the file path), and `submodule_search_locations` (non-`None` for packages). `importlib.util.spec_from_loader` builds one.
- A **loader** creates and runs the module: `create_module(spec)` (return `None` for the default) and `exec_module(module)`, which runs the code in `module.__dict__`.

To add a new source of modules, write a class with `find_spec` (a meta path finder) and `exec_module` (a loader), and put an instance at the front of `sys.meta_path`. A finder returns `None` for names it doesn't handle, so the rest of the chain still works. Always remove your finder again, and the modules it loaded, when you're done.

## Packages and relative imports

A **package** is a module that can contain submodules. On disk it's a folder with an `__init__.py` (which runs when the package is imported); in the import system, it's a module with a **`__path__`** attribute: a list of places to search for its submodules. When you import `shop.cart`, Python imports `shop` first, then asks the finders for `shop.cart`, passing `shop.__path__`.

A folder **without** `__init__.py` on `sys.path` becomes a **namespace package** (PEP 420): it has a `__path__` that can span several folders, so separately installed parts can share one package name.

**Relative imports** (`from . import cart`, `from .prices import price`, `from .. import utils`) are resolved using the importing module's **`__package__`** (the package it belongs to): `.` is that package, `..` its parent. That's why relative imports only work inside packages, and fail with "attempted relative import with no known parent package" in a script run directly (whose `__package__` is empty). Running a module with `python -m shop.cart` sets its package properly.

## Circular imports

Since a module is put in `sys.modules` **before** its code runs, a module that's still running can be imported by another. That's what makes **circular imports** possible: if `a` imports `b` and `b` imports `a`, then when `b` runs `import a` it receives the **partially initialised** module `a`, which only has the names defined above `a`'s `import b` line.

- `import a` followed by using `a.thing` **later** (inside a function) usually works, because by then `a` has finished.
- `from a import thing` at the top of `b` fails with `ImportError: cannot import name 'thing' from partially initialized module 'a' (most likely due to a circular import)`, because `thing` doesn't exist yet.

Fixes, best first: **restructure** (move what both need into a third module that imports neither); import **inside the function** that needs it; or use `import a` instead of `from a import ...` and look the name up when it's used. Type hints that cause cycles can be guarded with `if typing.TYPE_CHECKING:`.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run `python -X importtime -c "import json"` to see every module importing `json` loads and how long each took. Tomorrow is the Forge: a code checker built from this week's tools.